# ED Pathway Trainer — v2e_plus_fix
Self-contained: embeds `ed_core` (skeleton + XAI/SLA/Audit) and runs the full pipeline.

In [ ]:

# === Bootstrap ed_core (embedded) ===
import pathlib
bundle = {"ed_core/__init__.py": "\nfrom .config import load_config\nfrom .actions import ActionSpec, ActionSpace\nfrom .heads import Backbone, GRUBackbone, ActionHead, make_action_head\nfrom .calibrate import TempScaler, fit_temperature\nfrom .gate import GateConfig, gate_predictions\nfrom .policy import apply_policy\nfrom .bridge import BridgeAdapter\nfrom .eval import safety_utility, compute_confusion\n", "ed_core/config.py": "\nfrom __future__ import annotations\nfrom dataclasses import dataclass, field\nfrom typing import List, Dict, Any\nimport yaml, json\n\n@dataclass\nclass Config:\n    actions: List[Dict[str, Any]]\n    features: List[str]\n    backbone: Dict[str, Any] = field(default_factory=lambda: {\"type\":\"gru\",\"hidden\":64,\"dropout\":0.3})\n    head: Dict[str, Any] = field(default_factory=lambda: {\"type\":\"unified\"})\n    thresholds: Dict[str, float] = field(default_factory=dict)\n    gate: Dict[str, Any] = field(default_factory=lambda: {\"tau\":0.5, \"top_k\":1})\n    temperature: float = 1.0\n    seed: int = 1337\n\ndef load_config(path: str) -> Config:\n    if path.endswith(\".json\"):\n        data = json.load(open(path))\n    else:\n        data = yaml.safe_load(open(path))\n    return Config(**data)\n", "ed_core/actions.py": "\nfrom __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import List, Dict, Any\n\n@dataclass(frozen=True)\nclass ActionSpec:\n    code: str\n    label: str\n    critical: bool = False\n    params: Dict[str, Any] = None\n\nclass ActionSpace:\n    def __init__(self, specs: List[Dict[str, Any]]):\n        self.specs: List[ActionSpec] = [ActionSpec(**s) for s in specs]\n        self.idx: Dict[str,int] = {s.code:i for i,s in enumerate(self.specs)}\n        self.codes: List[str] = [s.code for s in self.specs]\n        self.labels: List[str] = [s.label for s in self.specs]\n        self.critical_mask = [int(s.critical) for s in self.specs]\n    def __len__(self): return len(self.specs)\n    def id(self, code: str) -> int: return self.idx[code]\n    def spec(self, i: int) -> ActionSpec: return self.specs[i]\n", "ed_core/heads.py": "\nfrom __future__ import annotations\nfrom dataclasses import dataclass\nfrom typing import Dict, Any, Optional, List\nimport torch, torch.nn as nn, torch.nn.functional as F\n\nclass Backbone(nn.Module):\n    def forward(self, x: torch.Tensor) -> torch.Tensor:  # [B, T, D] -> [B, H]\n        raise NotImplementedError\n\nclass GRUBackbone(Backbone):\n    def __init__(self, input_dim: int, hidden: int=64, dropout: float=0.3):\n        super().__init__()\n        self.gru = nn.GRU(input_dim, hidden, num_layers=1, batch_first=True)\n        self.drop = nn.Dropout(dropout)\n        self.hidden = hidden\n    def forward(self, x):\n        out, _ = self.gru(x)   # [B,T,H]\n        return self.drop(out[:,-1,:])  # [B,H]\n\nclass UnifiedHead(nn.Module):\n    def __init__(self, emb_dim: int, num_actions: int):\n        super().__init__()\n        self.fc = nn.Linear(emb_dim, num_actions)\n    def forward(self, h):  # [B,H] -> [B,K]\n        return self.fc(h)\n\nclass MultiHeadActions(nn.Module):\n    def __init__(self, emb_dim: int, num_actions: int):\n        super().__init__()\n        self.heads = nn.ModuleList([nn.Linear(emb_dim, 1) for _ in range(num_actions)])\n    def forward(self, h):  # [B,H] -> [B,K]\n        logits = [head(h) for head in self.heads]\n        return torch.cat(logits, dim=1)  # [B,K]\n\nclass ActionHead(nn.Module):\n    def __init__(self, backbone: Backbone, head_type: str, num_actions: int):\n        super().__init__()\n        self.backbone = backbone\n        self.num_actions = num_actions\n        if head_type == \"multi\":\n            self.classifier = MultiHeadActions(backbone.hidden, num_actions)\n        else:\n            self.classifier = UnifiedHead(backbone.hidden, num_actions)\n    def forward(self, x):  # [B,T,D] -> logits [B,K]\n        h = self.backbone(x)\n        logits = self.classifier(h)\n        return logits\n\ndef make_action_head(input_dim: int, num_actions: int, cfg: Dict[str,Any]) -> ActionHead:\n    bb = GRUBackbone(input_dim, hidden=cfg.get(\"backbone\",{}).get(\"hidden\",64),\n                     dropout=cfg.get(\"backbone\",{}).get(\"dropout\",0.3))\n    head_type = cfg.get(\"head\",{}).get(\"type\",\"unified\")\n    return ActionHead(bb, head_type=head_type, num_actions=num_actions)\n", "ed_core/calibrate.py": "\nfrom __future__ import annotations\nimport torch, torch.nn as nn, torch.nn.functional as F\n\nclass TempScaler(nn.Module):\n    def __init__(self): super().__init__(); self.logT = nn.Parameter(torch.zeros(1))\n    def forward(self, logits): return logits / (torch.exp(self.logT) + 1e-6)\n\n@torch.no_grad()\ndef fit_temperature(model: nn.Module, logits: torch.Tensor, targets: torch.Tensor, max_iter=200):\n    model.eval()\n    ts = TempScaler()\n    nll = nn.CrossEntropyLoss()\n    opt = torch.optim.LBFGS(ts.parameters(), lr=0.05, max_iter=max_iter, line_search_fn=\"strong_wolfe\")\n    def closure():\n        opt.zero_grad()\n        loss = nll(ts(logits), targets)\n        loss.backward()\n        return loss\n    opt.step(closure)\n    return ts\n", "ed_core/gate.py": "\nfrom __future__ import annotations\nfrom dataclasses import dataclass, field\nfrom typing import Dict, Any, Optional, Tuple\nimport numpy as np\n\n@dataclass\nclass GateConfig:\n    tau: float = 0.5\n    per_class: Dict[int, float] = field(default_factory=dict)\n    top_k: int = 1\n\ndef gate_predictions(probs: np.ndarray, cfg: GateConfig) -> Tuple[np.ndarray, Dict[str,Any]]:\n    N, K = probs.shape\n    th = np.full(K, cfg.tau, dtype=float)\n    for k, v in cfg.per_class.items():\n        th[int(k)] = float(v)\n    mask = probs >= th  # [N,K] broadcast\n    top = probs.argmax(1)\n    pred = np.where(mask[np.arange(N), top], top, 0)\n    topk_idx = np.argsort(-probs, axis=1)[:, :cfg.top_k]\n    return pred, {\"thresholds\": th.tolist(), \"topk\": topk_idx.tolist()}\n", "ed_core/policy.py": "\nfrom __future__ import annotations\nfrom typing import List, Dict, Any\nimport numpy as np\n\ndef apply_policy(pred_idx: np.ndarray,\n                 probs: np.ndarray,\n                 context: Dict[str,Any],\n                 actions: List[Dict[str,Any]]) -> List[Dict[str,Any]]:\n    out = []\n    for i, a_idx in enumerate(pred_idx.tolist()):\n        spec = actions[a_idx]\n        msg = {\"action_code\": spec[\"code\"], \"params\": spec.get(\"params\", {}), \"proba\": float(probs[i, a_idx])}\n        if spec[\"code\"] == \"ORDER_CT\":\n            approved = context.get(\"approval\", {}).get(\"ORDER_CT\", False)\n            if not approved or context.get(\"cap_stale\", False):\n                msg[\"requires_approval\"] = True\n                msg[\"blocked_reason\"] = \"CT needs approval and fresh capacity\"\n        if spec[\"code\"] == \"REQUEST_BED\" and context.get(\"cap_stale\", False):\n            msg[\"action_code\"] = \"VERIFY_CAPACITY\"\n            msg[\"note\"] = \"Capacity stale -> repoll before bed request\"\n        unc = context.get(\"uncertainty\", None)\n        if unc is not None:\n            msg[\"needs_check\"] = bool(unc[i] > np.quantile(unc, 0.75))\n        out.append(msg)\n    return out\n", "ed_core/bridge.py": "\nfrom __future__ import annotations\nfrom typing import Dict, Any, List\n\nclass BridgeAdapter:\n    def __init__(self, system: str = \"json\"):\n        self.system = system\n    def to_message(self, action: Dict[str,Any]) -> Dict[str,Any]:\n        code = action[\"action_code\"]\n        params = action.get(\"params\", {})\n        if code.startswith(\"ORDER_\"):\n            return {\"type\":\"ORDER\", \"modality\":code.replace(\"ORDER_\",\"\"), \"params\":params}\n        if code in (\"REQUEST_BED\", \"VERIFY_CAPACITY\"):\n            return {\"type\":\"BED_OP\", \"op\":code, \"params\":params}\n        if code == \"REQUEST_CONSULT\":\n            return {\"type\":\"CONSULT\", \"service\":params.get(\"service\",\"auto\")}\n        return {\"type\":\"INFO\", \"code\":code, \"params\":params}\n", "ed_core/eval.py": "\nfrom __future__ import annotations\nfrom typing import Dict, Any, Tuple\nimport numpy as np\nfrom sklearn.metrics import confusion_matrix, precision_recall_fscore_support\n\ndef safety_utility(y_true, y_pred, fp_cost=1.0, fn_cost=5.0) -> float:\n    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)\n    return (- fp_cost * np.sum((y_pred != y_true) & (y_true == 0))\n            - fn_cost * np.sum((y_pred != y_true) & (y_true != 0))) / len(y_true)\n\ndef compute_confusion(y_true, y_pred, K: int) -> Dict[str,Any]:\n    cm = confusion_matrix(y_true, y_pred, labels=list(range(K)))\n    prf = precision_recall_fscore_support(y_true, y_pred, labels=list(range(K)), zero_division=0)\n    return {\"cm\": cm.tolist(), \"precision\": prf[0].tolist(), \"recall\": prf[1].tolist(), \"f1\": prf[2].tolist()}\n", "ed_core/replay.py": "\nfrom __future__ import annotations\nimport argparse, json, numpy as np, torch\nfrom typing import Dict, Any\nfrom .config import load_config\nfrom .actions import ActionSpace\nfrom .heads import make_action_head\nfrom .calibrate import TempScaler\nfrom .gate import GateConfig, gate_predictions\nfrom .policy import apply_policy\nfrom .bridge import BridgeAdapter\nfrom .metrics import SLARecorder, safety_utility\nfrom .xai import shallow_attributions\nfrom .audit import AuditLog, export_anchor\nfrom .policy_tests import detect_duplicate_orders, detect_contradictions, stale_capacity_violation\nfrom .notifier import Notifier\n\ndef run_replay(cfg_path: str, events_path: str, model_path: str, out_path: str, audit_path: str=None):\n    cfg = load_config(cfg_path)\n    actions = ActionSpace(cfg.actions)\n    sla = SLARecorder()\n    audit = AuditLog(audit_path) if audit_path else None\n    notifier = Notifier()\n    events = [json.loads(line) for line in open(events_path)]\n    X = np.array([e[\"features\"] for e in events], dtype=float)\n    X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)\n    model = make_action_head(input_dim=X.shape[-1], num_actions=len(actions), cfg=cfg.__dict__)\n    model.load_state_dict(torch.load(model_path, map_location=\"cpu\"))\n    model.eval()\n    logits = model(X)\n    T = cfg.temperature\n    logits = logits / max(T, 1e-6)\n    probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()\n    gcfg = GateConfig(tau=cfg.gate.get(\"tau\",0.5),\n                      per_class={int(k):float(v) for k,v in cfg.thresholds.items()},\n                      top_k=int(cfg.gate.get(\"top_k\",1)))\n    pred, info = gate_predictions(probs, gcfg)\n    action_msgs = apply_policy(pred, probs, {\"cap_stale\": False, \"approval\": {\"ORDER_CT\": False}, \"uncertainty\": None},\n                               [s.__dict__ for s in actions.specs])\n    bridge = BridgeAdapter(system=\"json\")\n    feature_names = cfg.features\n    atts = shallow_attributions(model=lambda x: model(x), temp=lambda z: z/ max(cfg.temperature,1e-6), xb=X, feature_names=feature_names, top_k=5)\n    out = []\n    for i, m in enumerate(action_msgs):\n        msg = bridge.to_message(m)\n        msg['xai_top_features'] = atts[i]\n        out.append(msg)\n        if audit:\n            audit.append({'event_id': i, 'action': m, 'message': msg})\n    sla.add_sample()\n    sla.mark_duplicate_blocked(detect_duplicate_orders(out))\n    sla.mark_contradiction_blocked(detect_contradictions(out))\nout = [bridge.to_message(m) for m in action_msgs]\n    with open(out_path, \"w\") as f:\n        for rec in out: f.write(json.dumps(rec) + \"\\n\")\n    if audit:\n        anchor = export_anchor(audit_path, audit_path+'.anchor.json')\n        print('Anchor:', anchor)\n    print('SLA summary:', json.dumps(sla.summary()))\n    print(f\"Wrote {len(out)} messages ->\", out_path)\n\nif __name__ == \"__main__\":\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--config\", required=True)\n    ap.add_argument(\"--events\", required=True)\n    ap.add_argument(\"--model\", required=True)\n    ap.add_argument(\"--out\",    required=True)\n    args = ap.parse_args()\n    run_replay(args.config, args.events, args.model, args.out)\n", "ed_core/ed_config.example.yaml": "\nactions:\n  - {code: \"NO_OP\",            label: \"No action\", critical: false}\n  - {code: \"ORDER_ECG\",        label: \"Order ECG\", critical: false}\n  - {code: \"PERFORM_FAST\",     label: \"Perform FAST\", critical: true}\n  - {code: \"ORDER_LABS\",       label: \"Order Labs\", critical: false}\n  - {code: \"ORDER_XR\",         label: \"Order X-Ray\", critical: false}\n  - {code: \"ORDER_CT\",         label: \"Order CT\", critical: true}\n  - {code: \"REQUEST_CONSULT\",  label: \"Request consult\", critical: false}\n  - {code: \"REQUEST_BED\",      label: \"Request bed\", critical: true}\n\nfeatures: [\"minute_of_day\", \"cap_stale\", \"ems\", \"consult_delay_min\",\n           \"syn_chest_pain\", \"syn_polytrauma\", \"syn_neuro_deficit\", \"syn_other\"]\n\nbackbone: {type: \"gru\", hidden: 64, dropout: 0.3}\nhead: {type: \"unified\"}\n\nthresholds:\n  \"2\": 0.25\n  \"5\": 0.25\n  \"7\": 0.30\n\ngate: {tau: 0.5, top_k: 3}\ntemperature: 1.0\nseed: 1337\n", "ed_core/xai.py": "\nfrom __future__ import annotations\nfrom typing import List, Dict, Any, Tuple\nimport torch\nimport numpy as np\n\ndef shallow_attributions(model: torch.nn.Module,\n                         temp,\n                         xb: torch.Tensor,\n                         feature_names: List[str],\n                         top_k: int = 5) -> List[List[Tuple[str, float]]]:\n    model.eval()\n    xb = xb.clone().detach().requires_grad_(True)\n    logits = model(xb)\n    if isinstance(logits, (tuple, list)):\n        logits = logits[0]\n    logits = temp(logits)\n    pred = logits.argmax(dim=1)\n    grads_all = []\n    for i in range(xb.size(0)):\n        model.zero_grad(set_to_none=True)\n        xb.grad = None\n        logits[i, pred[i]].backward(retain_graph=True)\n        g = xb.grad[i]\n        sal = g.abs().sum(dim=0).detach().cpu().numpy()\n        grads_all.append(sal)\n    out = []\n    for sal in grads_all:\n        idx = np.argsort(-sal)[:min(top_k, len(sal))]\n        out.append([(feature_names[j], float(sal[j])) for j in idx])\n    return out\n", "ed_core/metrics.py": "\nfrom __future__ import annotations\nfrom typing import Dict, Any, Optional\nimport time, numpy as np\n\nclass SLARecorder:\n    def __init__(self):\n        self.t = {}\n        self.lat = {}\n        self.counters = {\"missed_critical\": 0, \"duplicates_blocked\": 0, \"contradictions_blocked\": 0}\n        self.samples = 0\n\n    def start(self, key: str): self.t[key] = time.time()\n    def end(self, key: str): self.lat[key] = self.lat.get(key, []) + [time.time() - self.t.get(key, time.time())]\n\n    def add_sample(self): self.samples += 1\n    def mark_missed_critical(self, n=1): self.counters[\"missed_critical\"] += n\n    def mark_duplicate_blocked(self, n=1): self.counters[\"duplicates_blocked\"] += n\n    def mark_contradiction_blocked(self, n=1): self.counters[\"contradictions_blocked\"] += n\n\n    def summary(self) -> Dict[str,Any]:\n        lat = {k: {\"p50\": float(np.median(v)), \"p95\": float(np.percentile(v,95)), \"mean\": float(np.mean(v))}\n               for k,v in self.lat.items()}\n        return {\"latency\": lat, \"counters\": self.counters, \"samples\": self.samples}\n\ndef safety_utility(y_true, y_pred, fp_cost=1.0, fn_cost=5.0) -> float:\n    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)\n    return (- fp_cost * np.sum((y_pred != y_true) & (y_true == 0))\n            - fn_cost * np.sum((y_pred != y_true) & (y_true != 0))) / len(y_true)\n", "ed_core/audit.py": "\nfrom __future__ import annotations\nfrom typing import Dict, Any, List\nimport os, json, hashlib\nfrom datetime import datetime, timezone\n\ndef _canon(obj: Dict[str,Any]) -> str:\n    return json.dumps(obj, sort_keys=True, separators=(\",\",\":\"))\n\nclass AuditLog:\n    def __init__(self, path: str):\n        self.path = path\n        self.prev_hash = None\n        if os.path.exists(path):\n            last = None\n            with open(path,\"r\") as f:\n                for line in f: last = line\n            if last:\n                try:\n                    rec = json.loads(last)\n                    self.prev_hash = rec.get(\"chain_hash\")\n                except Exception:\n                    self.prev_hash = None\n\n    def append(self, event: Dict[str,Any]) -> Dict[str,Any]:\n        ts = datetime.now(timezone.utc).isoformat()\n        rec = {\"ts\": ts, \"event\": event}\n        base = (self.prev_hash or \"\") + _canon(rec)\n        ch = hashlib.sha256(base.encode(\"utf-8\")).hexdigest()\n        out = {\"ts\": ts, \"event\": event, \"chain_hash\": ch}\n        with open(self.path, \"a\") as f:\n            f.write(json.dumps(out) + \"\\n\")\n        self.prev_hash = ch\n        return out\n\ndef merkle_root(records: List[str]) -> str:\n    if not records: return \"\"\n    layer = [bytes.fromhex(h) for h in records]\n    import hashlib\n    while len(layer) > 1:\n        nxt = []\n        for i in range(0, len(layer), 2):\n            a = layer[i]\n            b = layer[i+1] if i+1 < len(layer) else a\n            nxt.append(hashlib.sha256(a+b).digest())\n        layer = nxt\n    return layer[0].hex()\n\ndef export_anchor(audit_path: str, out_path: str):\n    hashes = []\n    with open(audit_path, \"r\") as f:\n        for line in f:\n            try:\n                rec = json.loads(line)\n                if \"chain_hash\" in rec: hashes.append(rec[\"chain_hash\"])\n            except Exception:\n                pass\n    root = merkle_root(hashes)\n    anchor = {\"anchored_at\": datetime.now(timezone.utc).isoformat(), \"merkle_root\": root, \"count\": len(hashes)}\n    with open(out_path, \"w\") as f:\n        json.dump(anchor, f, indent=2)\n    return anchor\n", "ed_core/bus.py": "\nfrom __future__ import annotations\nfrom typing import Callable, Dict, List, Any\nimport threading, queue\n\nclass InMemoryBus:\n    def __init__(self):\n        self.subs: Dict[str, List[Callable[[Any],None]]] = {}\n        self.q = queue.Queue()\n        self.running = True\n        self.thread = threading.Thread(target=self._run, daemon=True)\n        self.thread.start()\n\n    def publish(self, topic: str, msg: Any):\n        self.q.put((topic, msg))\n\n    def subscribe(self, topic: str, fn: Callable[[Any],None]):\n        self.subs.setdefault(topic, []).append(fn)\n\n    def _run(self):\n        while self.running:\n            try:\n                topic, msg = self.q.get(timeout=0.1)\n            except queue.Empty:\n                continue\n            for fn in self.subs.get(topic, []):\n                try: fn(msg)\n                except Exception: pass\n\n    def stop(self):\n        self.running = False\n        self.thread.join(timeout=1.0)\n", "ed_core/policy_tests.py": "\nfrom __future__ import annotations\nfrom typing import List, Dict, Any\n\ndef detect_duplicate_orders(actions: List[Dict[str,Any]]) -> int:\n    seen = set(); dup = 0\n    for a in actions:\n        key = (a.get(\"type\"), a.get(\"modality\") or a.get(\"op\"), str(a.get(\"params\",{})))\n        if key in seen: dup += 1\n        else: seen.add(key)\n    return dup\n\ndef detect_contradictions(actions: List[Dict[str,Any]]) -> int:\n    has_ct = any(a.get(\"type\") == \"ORDER\" and a.get(\"modality\") == \"CT\" for a in actions)\n    has_noop = any(a.get(\"type\") == \"INFO\" and a.get(\"code\") == \"NO_OP\" for a in actions)\n    return int(has_ct and has_noop)\n\ndef stale_capacity_violation(actions: List[Dict[str,Any]], cap_stale: bool) -> int:\n    if not cap_stale: return 0\n    return sum(1 for a in actions if (a.get(\"type\") == \"ORDER\" and a.get(\"modality\") == \"CT\") or (a.get(\"type\")==\"BED_OP\" and a.get(\"op\")==\"REQUEST_BED\"))\n", "ed_core/notifier.py": "\nfrom __future__ import annotations\nfrom typing import Dict, Any, List\nimport time\n\nclass Notifier:\n    def __init__(self):\n        self.queue: List[Dict[str,Any]] = []\n    def send(self, msg: Dict[str,Any]) -> bool:\n        self.queue.append({\"ts\": time.time(), \"msg\": msg, \"status\": \"queued\"})\n        return True\n"}
for rel, src in bundle.items():
    p = pathlib.Path(rel)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w", encoding="utf-8") as f:
        f.write(src)
print("Wrote", len(bundle), "files under ed_core/")


In [ ]:

import os, math, random, json, time, base64
from dataclasses import dataclass, field
from typing import List, Dict, Any, Tuple
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import train_test_split

from ed_core.eval import safety_utility
from ed_core.calibrate import TempScaler

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


In [ ]:

@dataclass
class GenConfig:
    target_pts_per_day_baseline: int = 100
    target_pts_per_day_stress: int = 160
    start_datetime: str = "2025-08-12 00:00:00"
    chest_pain_share_mean: float = 0.30
    chest_pain_share_sd: float = 0.08
    polytrauma_share_mean: float = 0.10
    neuro_share_mean: float = 0.08
    night_mult: tuple = (1.25, 1.4)
    weekend_mult: tuple = (1.15, 1.25)
    stale_prob_base: float = 0.20
    stale_burst_every_hours: tuple = (6, 12)
    stale_burst_min: tuple = (45, 90)
    missing_ts_prob: float = 0.05
    cancel_prob: float = 0.05
    code_flip_prob: float = 0.015
    include_ecg_fast_as_actions: bool = True

ACTIONS = ["NO_OP","ORDER_ECG","PERFORM_FAST","ORDER_LABS","ORDER_XR","ORDER_CT","REQUEST_CONSULT","REQUEST_BED"]
act2id = {a:i for i,a in enumerate(ACTIONS)}

def sample_daily_arrival_curve(total):
    base = np.array([1.0 + 0.3*np.sin((h-7)/24*2*np.pi) for h in range(24)])
    if random.random()<0.5:
        base = np.array([1.0 + 0.3*np.sin((h-17)/24*2*np.pi) for h in range(24)])
    base = base / base.sum()
    spikes = random.sample(range(24), k=random.randint(1,3))
    for s in spikes:
        base[s] *= random.uniform(1.4, 1.8)
    base = base / base.sum()
    lam = base * total
    arr = []
    for h in range(24):
        n = np.random.poisson(lam[h])
        arr += [h*60 + random.randint(0,59) for _ in range(n)]
    arr.sort()
    return arr

def vary_case_mix(mu, sd):
    cp = np.clip(np.random.normal(mu, sd), 0.05, 0.6)
    tr = np.clip(np.random.normal(GenConfig.polytrauma_share_mean, 0.05), 0.02, 0.2)
    ne = np.clip(np.random.normal(GenConfig.neuro_share_mean, 0.04), 0.02, 0.2)
    ot = max(0.0, 1.0 - cp - tr - ne)
    return {"chest_pain": cp, "polytrauma": tr, "neuro_deficit": ne, "other": ot}

class ResourceSystem:
    def __init__(self, cfg: GenConfig, is_weekend: bool, is_night: bool):
        nm = random.uniform(*cfg.night_mult) if is_night else 1.0
        wm = random.uniform(*cfg.weekend_mult) if is_weekend else 1.0
        self.lab_rate = 35.0 * nm * wm
        self.xr_rate  = 50.0 * nm * wm
        self.ct_rate  = 70.0 * nm * wm
        self.lab_q = 0; self.xr_q=0; self.ct_q=0
    def gamma_qtat(self, base):
        q_mult = 1.0 + 0.01*(self.lab_q + self.xr_q + self.ct_q)
        shape, scale = 2.0, base/2.0
        return np.random.gamma(shape, scale) * q_mult

def simulate_one_day(cfg: GenConfig, stress: bool, day_index: int):
    target = cfg.target_pts_per_day_stress if stress else cfg.target_pts_per_day_baseline
    arr_min = sample_daily_arrival_curve(target)
    mix = vary_case_mix(cfg.chest_pain_share_mean, cfg.chest_pain_share_sd)
    st = pd.Timestamp(cfg.start_datetime) + pd.Timedelta(days=day_index)
    next_burst = st + pd.Timedelta(hours=random.randint(*cfg.stale_burst_every_hours))
    burst_until = st
    rows = []
    for tmin in arr_min:
        ts = st + pd.Timedelta(minutes=int(tmin))
        hod = ts.hour
        is_weekend = ts.dayofweek >= 5
        is_night = (hod<7) or (hod>=20)
        rs = ResourceSystem(cfg, is_weekend, is_night)

        syn = random.choices(list(mix.keys()), weights=list(mix.values()))[0]
        ems = (random.random()<0.6)

        if ts >= next_burst:
            burst_len = random.randint(*cfg.stale_burst_min)
            burst_until = ts + pd.Timedelta(minutes=burst_len)
            next_burst = ts + pd.Timedelta(hours=random.randint(*cfg.stale_burst_every_hours))
        cap_stale = (random.random()<cfg.stale_prob_base) or (ts < burst_until)

        lab_submit = tmin + np.random.exponential(15) if random.random()<0.7 else None
        xr_submit  = tmin + np.random.exponential(25) if (syn!="other" and random.random()<0.25) else None
        ct_submit  = tmin + np.random.exponential(35) if (syn in ("polytrauma","neuro_deficit") and random.random()<0.25) else None

        lab_done = lab_submit + rs.gamma_qtat(rs.lab_rate) if lab_submit else None
        xr_done  = xr_submit  + rs.gamma_qtat(rs.xr_rate)  if xr_submit  else None
        ct_done  = ct_submit  + rs.gamma_qtat(rs.ct_rate)  if ct_submit  else None

        consult_delay = max(0, np.random.normal(30, 12)) if (
            (syn=="chest_pain" and random.random()<0.5) or
            (syn=="neuro_deficit" and random.random()<0.55) or
            (syn=="polytrauma" and random.random()<0.30)
        ) else 0.0

        duplicate_order = (random.random()<0.03)
        cancel = (random.random()<0.05)
        code_flip = (random.random()<0.015)
        negative_control = (random.random()<0.03)

        rows.append(dict(
            day_index=day_index, timestamp=ts.isoformat(), minute_of_day=int(tmin),
            daytype=("weekend_night" if is_weekend and is_night else
                     "weekend_day" if is_weekend else
                     "weekday_night" if is_night else "weekday_day"),
            syndrome=syn, consult_delay_min=float(consult_delay),
            lab_submit_min=lab_submit, lab_done_min=lab_done,
            xr_submit_min=xr_submit,   xr_done_min=xr_done,
            ct_submit_min=ct_submit,   ct_done_min=ct_done,
            cap_stale=bool(cap_stale), duplicate_order=duplicate_order,
            cancel=cancel, code_flip=code_flip,
            negative_control=negative_control, ems=bool(ems)
        ))
    return pd.DataFrame(rows)

def simulate_days(cfg: GenConfig, n_days=5, stress_ratio=0.35):
    frames = []
    for d in range(n_days):
        frames.append(simulate_one_day(cfg, stress=(random.random()<stress_ratio), day_index=d))
    df = pd.concat(frames, ignore_index=True)
    df.sort_values(["day_index","minute_of_day"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    print("Simulated rows:", len(df))
    return df

cfg = GenConfig()
df_raw = simulate_days(cfg, n_days=5, stress_ratio=0.35)
df_raw.head(3)


In [ ]:

X_COLS = ["minute_of_day","cap_stale","ems","consult_delay_min","syn_chest_pain","syn_polytrauma","syn_neuro_deficit","syn_other"]
for s in ["chest_pain","polytrauma","neuro_deficit","other"]:
    df_raw["syn_"+s] = (df_raw["syndrome"]==s).astype(int)

ACTIONS = ["NO_OP","ORDER_ECG","PERFORM_FAST","ORDER_LABS","ORDER_XR","ORDER_CT","REQUEST_CONSULT","REQUEST_BED"]
act2id = {a:i for i,a in enumerate(ACTIONS)}
A_WIN = {"ORDER_ECG":10,"PERFORM_FAST":15,"ORDER_CT":45,"ORDER_XR":30,"ORDER_LABS":20}
def within(submit, t, w): return (submit is not None) and (0 <= (submit - t) <= w)

def derive_label(row):
    if row["negative_control"]: return act2id["NO_OP"]
    t = row["minute_of_day"]
    if True: # include_ecg_fast_as_actions
        if row["syndrome"]=="chest_pain" and random.random()<0.7: return act2id["ORDER_ECG"]
        if row["syndrome"]=="polytrauma" and random.random()<0.7: return act2id["PERFORM_FAST"]
    if within(row.get("ct_submit_min"), t, A_WIN["ORDER_CT"]): return act2id["ORDER_CT"]
    if within(row.get("xr_submit_min"), t, A_WIN["ORDER_XR"]): return act2id["ORDER_XR"]
    if row["syndrome"] in ("chest_pain","other"):
        if within(row.get("lab_submit_min"), t, A_WIN["ORDER_LABS"]) and (random.random()<0.5): return act2id["ORDER_LABS"]
    else:
        if within(row.get("lab_submit_min"), t, A_WIN["ORDER_LABS"]) and (random.random()<0.25): return act2id["ORDER_LABS"]
    if row["consult_delay_min"]>25 and random.random()<0.6: return act2id["REQUEST_CONSULT"]
    if (row.get("ct_done_min") or row.get("xr_done_min") or row.get("lab_done_min")) and not row["cap_stale"]:
        if random.random()<0.25: return act2id["REQUEST_BED"]
    return act2id["NO_OP"]

df = df_raw.copy()
df["y"] = df.apply(derive_label, axis=1)

X = df[X_COLS].copy()
X["cap_stale"] = X["cap_stale"].astype(int)
X["ems"] = X["ems"].astype(int)

y = df["y"].astype(int).values
print("Features:", X_COLS)
print("Class counts:", dict(zip(ACTIONS, np.bincount(y, minlength=len(ACTIONS)).tolist())))


In [ ]:

X_np = X.values.astype(np.float32)
split = int(len(X_np)*0.85)
X_train, X_val = X_np[:split], X_np[split:]
y_train, y_val = y[:split], y[split:]

train_ds = TensorDataset(torch.tensor(X_train).unsqueeze(1), torch.tensor(y_train))
val_ds   = TensorDataset(torch.tensor(X_val).unsqueeze(1),   torch.tensor(y_val))
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=64)
print("Train/Val sizes:", X_train.shape, X_val.shape)


In [ ]:

class GRUHead(nn.Module):
    def __init__(self, input_dim, hidden=96, num_classes=len(ACTIONS), p_drop=0.35):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(p_drop)
        self.fc_multi = nn.Linear(hidden, num_classes)
        self.fc_bin = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.gru(x)
        h = self.dropout(out[:, -1, :])
        return self.fc_multi(h), self.fc_bin(h)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma; self.alpha = alpha; self.reduction = reduction
    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, reduction='none')
        pt = torch.exp(-ce)
        loss = ((1-pt)**self.gamma) * ce
        if self.alpha is not None:
            at = self.alpha.to(logits.device)[target]
            loss = at * loss
        return loss.mean() if self.reduction=='mean' else loss.sum()

def entropy_reg(probs, strength=7e-4):
    return strength * (- (probs.clamp_min(1e-8)*probs.clamp_min(1e-8).log()).sum(dim=1).mean())

counts = np.bincount(y_train, minlength=len(ACTIONS)).astype(float)
freq = counts / counts.sum()
inv = 1.0 / np.clip(freq, 1e-4, 1.0)
alpha_np = inv / inv.sum(); alpha_np[0] *= 0.5
alpha = torch.tensor(alpha_np, dtype=torch.float32)

pos_weight = torch.tensor([(len(y_train)-(y_train!=0).sum())/max(1,(y_train!=0).sum())], dtype=torch.float32)

model = GRUHead(input_dim=X_train.shape[1], hidden=96, num_classes=len(ACTIONS), p_drop=0.35)
model.to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=5e-4)
criterion_focal = FocalLoss(gamma=2.0, alpha=alpha, reduction='mean')
bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))

best_val = 1e9
EPOCHS = 12
for ep in range(EPOCHS):
    model.train(); total=0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits_mc, logit_bin = model(xb)
        yb_bin = (yb!=0).float()
        loss = criterion_focal(logits_mc, yb)              + 0.5 * bce(logit_bin.squeeze(1), yb_bin.to(DEVICE))              + entropy_reg(F.softmax(logits_mc, dim=-1))
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()*len(yb)
    model.eval(); vloss=0.0
    with torch.no_grad():
        for xb, yb in val_dl:
            logits_mc, logit_bin = model(xb.to(DEVICE))
            yb_bin = (yb!=0).float().to(DEVICE)
            vloss += (F.cross_entropy(logits_mc, yb.to(DEVICE)) + 0.5*bce(logit_bin.squeeze(1), yb_bin)).item()*len(yb)
    print(f"Epoch {ep+1}: train={total/len(X_train):.4f} val={vloss/len(X_val):.4f}")
torch.save(model.state_dict(), "ga_model.pt")
print(model)


In [ ]:

model.eval()
logits_val, y_true_val = [], []
with torch.no_grad():
    for xb, yb in val_dl:
        logits_mc, _ = model(xb.to(DEVICE))
        logits_val.append(logits_mc.cpu()); y_true_val.append(yb)
logits_val = torch.cat(logits_val, dim=0)
y_true_val = torch.cat(y_true_val, dim=0)
ts = TempScaler()
optim = torch.optim.LBFGS(ts.parameters(), lr=0.05, max_iter=200, line_search_fn="strong_wolfe")
nll = nn.CrossEntropyLoss()
def closure():
    optim.zero_grad()
    loss = nll(ts(logits_val), y_true_val)
    loss.backward(); return loss
optim.step(closure)
T = float(torch.exp(ts.logT).item())
json.dump({"T": T}, open("calibration.json","w"))
print("Temperature:", T)
def apply_temp(logits): return logits / max(T, 1e-6)
temp_module = lambda z: apply_temp(z)


In [ ]:

# === Robust temperature calibration (grid search with guardrails) ===
import numpy as np, json, torch, torch.nn.functional as F

nll = torch.nn.CrossEntropyLoss()

with torch.no_grad():
    base_nll = nll(logits_val, y_true_val).item()

Ts = np.exp(np.linspace(np.log(0.5), np.log(3.0), 41))  # clamp T to a sane range
best_T, best_nll = 1.0, base_nll
for t in Ts:
    nll_t = nll(logits_val/float(t), y_true_val).item()
    if nll_t < best_nll - 1e-6:
        best_T, best_nll = float(t), float(nll_t)

# If calibration doesn't improve NLL, fall back to T=1.0
T = best_T if best_nll < base_nll - 1e-6 else 1.0

# Re-define temp application for downstream cells
def apply_temp(logits): return logits / max(T, 1e-6)
temp_module = lambda z: apply_temp(z)

# Optional: report ECE pre/post
def ece_from_logits(logits, y, bins=15):
    probs = F.softmax(logits, dim=-1).cpu().numpy()
    conf = probs.max(1); pred = probs.argmax(1); y_np = y.cpu().numpy()
    acc = (pred==y_np).astype(np.float32)
    bounds = np.linspace(0,1,bins+1)
    ece = 0.0
    for i in range(bins):
        m = (conf > bounds[i]) & (conf <= bounds[i+1])
        if m.sum() > 0:
            ece += abs(acc[m].mean() - conf[m].mean()) * (m.mean())
    return float(ece)

ece_pre = ece_from_logits(logits_val, y_true_val)
ece_post = ece_from_logits(logits_val/ max(T,1e-6), y_true_val)

json.dump({"T": T, "base_nll": base_nll, "best_nll": best_nll, "ece_pre": ece_pre, "ece_post": ece_post},
          open("calibration.json","w"))
print(f"[Calib] T={T:.3f} baseNLL={base_nll:.3f} bestNLL={best_nll:.3f} | ECE pre={ece_pre:.3f} post={ece_post:.3f}")


In [ ]:

probs_val = []
with torch.no_grad():
    for xb, yb in val_dl:
        logits_mc, _ = model(xb.to(DEVICE))
        p = torch.softmax(temp_module(logits_mc), dim=-1).cpu().numpy()
        probs_val.append(p)
probs_val = np.concatenate(probs_val, axis=0)
y_true_val_np = y_true_val.numpy()

def sweep_tau(probs, y_true, taus=np.linspace(0.3,0.9,13)):
    best = None
    for tau in taus:
        pred = probs.argmax(1).copy()
        pred[probs.max(1) < tau] = 0
        util = safety_utility(y_true, pred, fp_cost=1.0, fn_cost=5.0)
        if (best is None) or (util>best["utility"]):
            best = {"tau": float(tau), "utility": float(util), "acc": float((pred==y_true).mean())}
    return best

best_tau = sweep_tau(probs_val, y_true_val_np)
json.dump({"best_tau": best_tau["tau"], "utility": best_tau["utility"]}, open("gate_tau.json","w"))
print(f"Best tau: {best_tau['tau']} utility: {best_tau['utility']:.3f}")

preds_dump = [{"idx": int(i), "probs": p.tolist()} for i,p in enumerate(probs_val)]
json.dump(preds_dump, open("gru_preds.json","w"))
print("Saved gru_preds.json with", len(preds_dump), "records.")


In [ ]:

import matplotlib.pyplot as plt
pred_tau = probs_val.argmax(1).copy()
pred_tau[probs_val.max(1) < best_tau["tau"]] = 0

def plot_cm(cm, labels, title, path):
    fig = plt.figure(figsize=(6,6)); ax = fig.add_subplot(111)
    im = ax.imshow(cm, interpolation='nearest')
    ax.set_title(title); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right'); ax.set_yticklabels(labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i,j]), ha='center', va='center', fontsize=8)
    fig.tight_layout(); fig.savefig(path); plt.close(fig)

labels = ACTIONS
cm_overall = confusion_matrix(y_true_val_np, pred_tau, labels=list(range(len(labels))))
plot_cm(cm_overall, labels, "Confusion matrix - overall (val)", "cm_overall_v5.png")
print("Saved cm_overall_v5.png")

val_idx = np.arange(len(df))[int(len(df)*0.85):]
daytypes = df.iloc[val_idx]["daytype"].values
hours = pd.to_datetime(df.iloc[val_idx]["timestamp"]).dt.hour.values
loadq = pd.qcut(hours, q=4, labels=False, duplicates="drop")
for dt in np.unique(daytypes):
    mask = (daytypes==dt)
    if mask.sum()<2: continue
    cm = confusion_matrix(y_true_val_np[mask], pred_tau[mask], labels=list(range(len(labels))))
    plot_cm(cm, labels, f"CM ({dt})", f"cm_daytype_{dt}.png")
for q in np.unique(loadq):
    mask = (loadq==q)
    if mask.sum()<2: continue
    cm = confusion_matrix(y_true_val_np[mask], pred_tau[mask], labels=list(range(len(labels))))
    plot_cm(cm, labels, f"CM (load q={q})", f"cm_loadq_{q}.png")
print("Saved stratified CM images.")


In [ ]:

import base64, json, os
from sklearn.metrics import classification_report, accuracy_score

def emb_png(path):
    with open(path, "rb") as f:
        return "data:image/png;base64," + base64.b64encode(f.read()).decode("ascii")

acc = accuracy_score(y_true_val_np, pred_tau)
report = classification_report(y_true_val_np, pred_tau, labels=list(range(len(ACTIONS))), 
                               target_names=ACTIONS, output_dict=True, zero_division=0)
tau = json.load(open("gate_tau.json"))["best_tau"]

html = f"""<!doctype html>
<html><head><meta charset="utf-8"><title>ED PoC – Brief v5 (DE)</title>
<style>body{{font-family:sans-serif; margin:24px}} .card{{border:1px solid #ddd; padding:16px; margin:12px 0; border-radius:10px}}</style>
</head><body>
<h1>ED Coordination – Clinician Brief v5</h1>
<div class="card"><h2>Übersicht</h2>
<p>Accuracy: {acc:.3f} · Tau: {tau}</p>
<img src="{emb_png('cm_overall_v5.png')}" style="max-width:480px"/>
</div>
<div class="card"><h2>Per-Klasse</h2>
<pre>{json.dumps(report, indent=2)}</pre>
</div>
</body></html>"""

open("ED_PoC_Clinician_Brief_DE_v5.html","w",encoding="utf-8").write(html)
print("Rendered ED_PoC_Clinician_Brief_DE_v5.html")


In [ ]:

def mc_probs(xb, n=20):
    model.train()
    with torch.no_grad():
        ps = []
        for _ in range(n):
            logits_mc, _ = model(xb.to(DEVICE))
            ps.append(torch.softmax(temp_module(logits_mc), dim=-1).cpu().numpy())
    return np.stack(ps, axis=0)

Ps, Ys = [], []
for xb, yb in val_dl:
    Ps.append(mc_probs(xb, n=20)); Ys.append(yb.numpy())
P = np.concatenate(Ps, axis=1); Y = np.concatenate(Ys, axis=0)
Pm = P.mean(0); Pv = P.var(0).mean(1)

tau_star = best_tau["tau"]
unc_q = 0.75
unc_th = np.quantile(Pv, unc_q)
pred_mc = Pm.argmax(1)
pred_mc[(Pm.max(1) < tau_star) | (Pv > unc_th)] = 0

acc_mc = accuracy_score(Y, pred_mc)
util_mc = safety_utility(Y, pred_mc)
json.dump({"acc": float(acc_mc), "util": float(util_mc), "unc_q": float(unc_q)}, open("mc_gate.json","w"))
print("MC-dropout gated acc:", acc_mc, "util:", util_mc)

html = open("ED_PoC_Clinician_Brief_DE_v5.html","r",encoding="utf-8").read()
mc = json.load(open("mc_gate.json"))
section = f"""<div class="card"><h2>MC-Dropout Gating</h2>
<p>unc_q={mc['unc_q']:.2f} · acc={mc['acc']:.3f} · util={mc['util']:.3f}</p></div>"""
html = html.replace("</body></html>", section + "\n</body></html>")
open("ED_PoC_Clinician_Brief_DE_v5.html","w",encoding="utf-8").write(html)
print("Brief updated with MC section.")


In [ ]:

from IPython.display import FileLink, display
for f in ["ga_model.pt","calibration.json","gate_tau.json","gru_preds.json",
          "cm_overall_v5.png","ED_PoC_Clinician_Brief_DE_v5.html"]:
    if os.path.exists(f):
        display(FileLink(f))
